# 01 — Exploration des données

**Objectif :** comprendre les trois tables nettoyées avant de construire le modèle en étoile (semaine 2).

Prérequis : avoir lancé `python src/ingest.py` puis `python src/clean.py`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data/processed")
pdc = pd.read_parquet(DATA / "points_recharge.parquet")
vehicules = pd.read_parquet(DATA / "vehicules.parquet")
communes = pd.read_parquet(DATA / "communes.parquet")

# Les dates sont stockées au format DATE (pour BigQuery) : conversion pour pandas
pdc["date_mise_en_service"] = pd.to_datetime(pdc["date_mise_en_service"])
vehicules["date_arrete"] = pd.to_datetime(vehicules["date_arrete"])

print(f"Points de recharge : {len(pdc):,}")
print(f"Lignes véhicules   : {len(vehicules):,}")
print(f"Communes           : {len(communes):,}")

## 1. Points de recharge : puissance et implantation

In [ ]:
ax = pdc["palier_puissance"].value_counts(sort=False).plot.barh(title="Points de recharge par palier de puissance")
ax.set_xlabel("Nombre de points")
plt.tight_layout()

In [ ]:
pdc["implantation_station"].value_counts(normalize=True).mul(100).round(1).to_frame("% des points")

## 2. Rythme de déploiement des points de recharge

In [ ]:
deploiement = (pdc.dropna(subset=["date_mise_en_service"])
                  .groupby(pdc["date_mise_en_service"].dt.year).size()
                  .cumsum())
ax = deploiement.plot(marker="o", title="Points de recharge publics en service (cumul, date connue)")
ax.set_xlabel("Année"); ax.set_ylabel("Points en service")
plt.tight_layout()

## 3. Évolution du parc de voitures électriques

In [ ]:
national = vehicules.groupby("date_arrete")[["nb_vp_electriques", "nb_vp_total"]].sum()
national["part_ve_%"] = national["nb_vp_electriques"] / national["nb_vp_total"] * 100
ax = national["nb_vp_electriques"].plot(title="Voitures électriques rechargeables immatriculées (France)")
ax.set_ylabel("Véhicules")
plt.tight_layout()
national.tail()

## 4. Premier aperçu de la question : ratio de tension par département

Dernier trimestre disponible, véhicules électriques / points de recharge publics.
⚠️ Résultat préliminaire : le biais des flottes d'entreprise (voir README) sera traité en semaine 2.

In [ ]:
dernier = vehicules["date_arrete"].max()
ve_dep = (vehicules[vehicules["date_arrete"] == dernier]
          .merge(communes[["code_insee", "code_departement"]], on="code_insee")
          .groupby("code_departement")["nb_vp_electriques"].sum())
pdc_dep = (pdc.merge(communes[["code_insee", "code_departement"]], on="code_insee")
              .groupby("code_departement").size())
pop_dep = communes.groupby("code_departement")["population"].sum()

dep = pd.DataFrame({"ve": ve_dep, "points_recharge": pdc_dep, "population": pop_dep}).dropna()
dep["ve_par_point"] = dep["ve"] / dep["points_recharge"]
dep["points_pour_10k_hab"] = dep["points_recharge"] / dep["population"] * 10_000

print(f"Trimestre : {dernier:%m/%Y}")
dep.sort_values("ve_par_point", ascending=False).head(10).round(1)

## 5. Notes pour la suite

- Vérifier les départements en tête du classement : effet flottes d'entreprise ?
- Décider du traitement des points sans date de mise en service pour l'analyse temporelle.
- Définir le grain de la table de faits véhicules (commune × trimestre) et des dimensions.